# SpatialMETA parameter benchmarking

## Imports

In [1]:
import seaborn as sns
import spatialmeta as smt
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

import time
import psutil
import os

from memory_profiler import memory_usage

## Loading data

In [2]:
joint_adata = smt.data.load_adata(
    sample_name="Y7_T_raw",
    modality="joint"
)

/home/mnxsybi_lumc/.conda/envs/spatial_clone_param/lib/python3.9/site-packages/spatialmeta/data/./datasets/adata_joint_Y7_T_raw_raw.h5ad


In [3]:
type(joint_adata)

anndata._core.anndata.AnnData

## Preprocessing

In [4]:
from memory_profiler import memory_usage
import time

def run_preprocessing(joint_adata):
    joint_adata = smt.pp.removeHSP_MT_RPL_DNAJ(joint_adata)

    joint_adata.layers["counts"] = joint_adata.X.copy()

    smt.pp.normalize_total_joint_adata_sm_st(joint_adata,
                            target_sum_SM=1e4,
                            target_sum_ST=1e4)

    joint_adata.layers["normalized"] = joint_adata.X.copy()

    joint_adata.raw = joint_adata

    smt.pp.spatial_variable_joint_adata_sm_st(joint_adata,
                                            n_top_genes = 2000,
                                            n_top_metabolites = 800,
                                            add_key = "highly_variable_moranI")

    joint_adata = joint_adata[:,joint_adata.var.highly_variable_moranI]

    joint_adata.write_h5ad("SpatialMETA/data/Y7_T_adata_joint_hvf2800.h5ad")

    return "test"

start_time = time.perf_counter()

mem_usage, result = memory_usage(
    (run_preprocessing, (joint_adata,)),
    retval=True,
    interval=0.1,
    max_usage=True
)

end_time = time.perf_counter()

print(f"Runtime: {end_time - start_time:.2f} seconds")
print(f"Peak RAM usage: {mem_usage:.2f} MiB")
#print(result)

Runtime: 1.98 seconds
Peak RAM usage: 1764.72 MiB


## CVAE model

In [5]:
from memory_profiler import memory_usage
import time

def run_CVAE(joint_adata):
    joint_adata = sc.read_h5ad("SpatialMETA/data/Y7_T_adata_joint_hvf2800.h5ad")

    joint_adata.X = joint_adata.layers["counts"]

    smt.pp.normalize_total_joint_adata_sm_st(
        joint_adata,
        target_sum_SM=1e3,
        target_sum_ST=None
    )

    model = smt.model.ConditionalVAESTSM(
        joint_adata,
        device='cpu', # Small change to CPU instead of CUDA
        reconstruction_method_sm='g',
        reconstruction_method_st='zinb',
    )

    loss_dict = model.fit(
        max_epoch=64,
        lr=1e-3,
        mode='single'
    )

    return joint_adata, model, loss_dict

start_time = time.perf_counter()

mem_usage, joint_adata, model, loss_dict = memory_usage(
    (run_CVAE, (joint_adata,)),
    retval=True,
    interval=0.1,
    max_usage=True
)

end_time = time.perf_counter()

print(f"Runtime: {end_time - start_time:.2f} seconds")
print(f"Peak RAM usage: {mem_usage:.2f} MiB")

Epoch 2:   3%|▎         | 2/64 [01:26<45:13, 43.77s/it, reconst_sm=1.42e+03, reconst_st=3.88e+02, reconst_sm_corr=1.24e+03, reconst_st_corr=3.88e+02, kldiv=7.24e-01, total_loss=6.54e+03, mmd_loss=0.00e+00]Process MemTimer-2:
Traceback (most recent call last):
  File "/home/mnxsybi_lumc/.conda/envs/spatial_clone_param/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/home/mnxsybi_lumc/.conda/envs/spatial_clone_param/lib/python3.9/site-packages/memory_profiler.py", line 262, in run
    stop = self.pipe.poll(self.interval)
  File "/home/mnxsybi_lumc/.conda/envs/spatial_clone_param/lib/python3.9/multiprocessing/connection.py", line 257, in poll
    return self._poll(timeout)
  File "/home/mnxsybi_lumc/.conda/envs/spatial_clone_param/lib/python3.9/multiprocessing/connection.py", line 424, in _poll
    r = wait([self], timeout)
  File "/home/mnxsybi_lumc/.conda/envs/spatial_clone_param/lib/python3.9/multiprocessing/connection.py", line 931, in wait
  

KeyboardInterrupt: 